In [2]:
import os
import librosa
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from tqdm import tqdm

# CONFIGURATION
SR = 16000
FRAME_DURATION = 4  # in seconds
N_MFCC = 39
N_CLUSTERS = 100
RANDOM_STATE = 42
OUTPUT_CSV = 'CMDC_BoAWMFCC_features.csv'

# === Global lists ===
features_list = []
failed_files = []

# === STEP 1: Extract MFCCs from each frame ===
def extract_mfcc_frames(audio_path, frame_duration=4, sr=16000, n_mfcc=39):
    try:
        y, _ = librosa.load(audio_path, sr=sr)
        frame_len = sr * frame_duration
        num_frames = len(y) // frame_len
        frame_mfccs = []

        for i in range(num_frames):
            start = i * frame_len
            end = start + frame_len
            frame = y[start:end]
            if len(frame) == frame_len:
                mfcc_feat = librosa.feature.mfcc(y=frame, sr=sr, n_mfcc=n_mfcc).T
                frame_mfccs.append(mfcc_feat)
        return frame_mfccs
    except Exception as e:
        print(f"❌ Error processing {audio_path}: {e}")
        return []

# === STEP 2: Directory Processing ===
def process_directory_by_frames(directory, label, features_list, frame_duration=4):
    total_files = sum(len(files) for _, _, files in os.walk(directory))
    with tqdm(total=total_files, desc=f"Processing {directory}") as pbar:
        for subdir, _, files in os.walk(directory):
            for file in files:
                if file.lower().endswith('.wav'):
                    file_path = os.path.join(subdir, file)
                    mfcc_frames = extract_mfcc_frames(file_path, frame_duration=frame_duration)
                    if not mfcc_frames:
                        failed_files.append(file_path)
                        pbar.update(1)
                        continue
                    for mfcc_frame in mfcc_frames:
                        features_list.append((mfcc_frame, label))
                    pbar.update(1)

# === STEP 3: Create Codebook ===
def create_codebook(mfcc_data, n_clusters=100):
    print("🔄 Fitting KMeans codebook on MFCC vectors...")
    all_vectors = np.vstack(mfcc_data)
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init='auto')
    kmeans.fit(all_vectors)
    return kmeans

# === STEP 4: BoAW Feature Extraction ===
def extract_boaw_features(mfcc_frame, kmeans_model):
    cluster_ids = kmeans_model.predict(mfcc_frame)
    hist, _ = np.histogram(cluster_ids, bins=np.arange(kmeans_model.n_clusters + 1))
    return hist / np.sum(hist)

# === STEP 5: Main execution ===
def main():
    # Process CMDC data
    process_directory_by_frames(r"C:\Users\Administrator\PycharmProjects\CMDC\Processed_Audios\silence_removed_HC", 0, features_list)
    process_directory_by_frames(r"C:\Users\Administrator\PycharmProjects\CMDC\Processed_Audios\silence_removed_MDD", 1, features_list)

    print(f"\n✅ Total valid frames: {len(features_list)}")
    print(f"❌ Total failed files: {len(failed_files)}")

    # Flatten MFCCs for codebook training
    all_mfcc_vectors = [vec for mfcc_seq, _ in features_list for vec in mfcc_seq]
    if len(all_mfcc_vectors) == 0:
        print("🚫 No MFCC vectors found. Please check the data.")
        return

    # Train KMeans codebook
    kmeans = create_codebook(all_mfcc_vectors, n_clusters=N_CLUSTERS)

    # Convert all frames to BoAW features
    X, y = [], []
    for mfcc_frame, label in tqdm(features_list, desc="Extracting BoAW features"):
        boaw_vector = extract_boaw_features(mfcc_frame, kmeans)
        X.append(boaw_vector)
        y.append(label)

    # Save as DataFrame
    df = pd.DataFrame(X)
    df['label'] = y
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Saved BoAWMFCC features to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()


Processing C:\Users\Administrator\PycharmProjects\CMDC\Processed_Audios\silence_removed_HC: 100%|█| 617/617 [01:33<00:0
Processing C:\Users\Administrator\PycharmProjects\CMDC\Processed_Audios\silence_removed_MDD: 100%|█| 305/305 [00:21<00:



✅ Total valid frames: 8197
❌ Total failed files: 32
🔄 Fitting KMeans codebook on MFCC vectors...


Extracting BoAW features: 100%|███████████████████████████████████████████████████| 8197/8197 [00:25<00:00, 323.88it/s]



✅ Saved BoAWMFCC features to: CMDC_BoAWMFCC_features.csv
